In [1]:
import json
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT_DIR = Path("/home/yangdejin/nlpcc/nlpcc_task2")
DATA_DIR = ROOT_DIR / "data"
OUTPUTS_DIR = ROOT_DIR / "outputs"
DEV_FILE = DATA_DIR / "dev.jsonl"


TRACK2_MODEL_WAY = 'dpo'
TRACK2_MODEL_NAME = "qwen35_9b"
TRACK2_MODEL_DIR = OUTPUTS_DIR / "track2" / TRACK2_MODEL_NAME / TRACK2_MODEL_WAY
EVAL_OUT_DIR = OUTPUTS_DIR / 'track2' / TRACK2_MODEL_NAME /'eval'/ TRACK2_MODEL_WAY
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
TRACK2_BASE_MODEL = "Qwen/Qwen3.5-9B"

TRACK1_MODEL_DIR = OUTPUTS_DIR / 'track1' / 'encoder' / 'bert_scl_optimized'
TRACK1_BASE_MODEL = "bert-base-uncased"

MAX_PROMPT_LENGTH = 640
MAX_NEW_TOKENS = 96
TRACK1_MAX_LENGTH = 512
TRACK1_BATCH_SIZE = 32

NUM_CANDIDATES = 4
TEMPERATURE = 0.7
TOP_P = 0.9

EXPERIMENT_NAME = f"{TRACK2_MODEL_WAY}_sample{NUM_CANDIDATES}"
USE_CACHE = True
GENERATION_FILE = EVAL_OUT_DIR / f'{TRACK2_MODEL_NAME}_dev_generations.jsonl'
DETAIL_FILE = EVAL_OUT_DIR / f'{TRACK2_MODEL_NAME}_track1_eval_details.csv'
print('DEV_FILE:', DEV_FILE)
print('TRACK2_MODEL_DIR:', TRACK2_MODEL_DIR)
print('TRACK1_MODEL_DIR:', TRACK1_MODEL_DIR)
print('GENERATION_FILE:', GENERATION_FILE)

DEV_FILE: /home/yangdejin/nlpcc/nlpcc_task2/data/dev.jsonl
TRACK2_MODEL_DIR: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/dpo
TRACK1_MODEL_DIR: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track1/encoder/bert_scl_optimized
GENERATION_FILE: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/eval/dpo/qwen35_9b_dev_generations.jsonl


In [2]:
# Cell 2: Official values, IO, prompt builder

VALUE_DEFINITIONS = {
    'Self-direction–thought': "Freedom to cultivate one's own ideas and abilities.",
    'Self-direction–action': "Freedom to determine one's own actions.",
    'Stimulation': 'Excitement, novelty, and change.',
    'Hedonism': 'Pleasure and sensuous gratification.',
    'Achievement': 'Success according to social standards.',
    'Power–dominance': 'Power through exercising control over people.',
    'Power–resources': 'Power through control of material and social resources.',
    'Face': "Security and power through maintaining one's public image and avoiding humiliation.",
    'Security–personal': "Safety in one's immediate environment.",
    'Security–societal': 'Safety and stability in the wider society.',
    'Tradition': 'Maintaining and preserving cultural, family, or religious traditions.',
    'Conformity–rules': 'Compliance with rules, laws, and formal obligations.',
    'Conformity–interpersonal': 'Avoidance of upsetting or harming other people.',
    'Humility': "Recognizing one's insignificance in the larger scheme of things.",
    'Benevolence–dependability': 'Being a reliable and trustworthy member of the ingroup.',
    'Benevolence–caring': 'Devotion to the welfare of ingroup members.',
    'Universalism–concern': 'Commitment to equality, justice, and protection for all people.',
    'Universalism–nature': 'Preservation of the natural environment.',
    'Universalism–tolerance': 'Acceptance and understanding of those who are different from oneself.',
}
VALUE_LABELS = list(VALUE_DEFINITIONS.keys())
label2id = {v: i for i, v in enumerate(VALUE_LABELS)}
id2label = {i: v for i, v in enumerate(VALUE_LABELS)}

TASK_INSTRUCTION = (
    'You are given a scenario, a question, and a target human value. '
    'Generate one concise, meaningful response that answers the question, '
    'fits the scenario, and naturally aligns with the target value.'
)
def normalize_space(text):
    return re.sub(r'\s+', ' ', str(text or '')).strip()

def build_prompt(row, use_value_definition=True):
    scenario = normalize_space(row.get('Scenario', ''))
    question = normalize_space(row.get('Question', ''))
    value = normalize_space(row.get('Value', ''))
    parts = [
        TASK_INSTRUCTION,
        f'Scenario:\n{scenario}',
        f'Question:\n{question}',
        f'Target value:\n{value}',
    ]
    if use_value_definition:
        parts.append(f'Target value definition:\n{VALUE_DEFINITIONS.get(value, "")}')
    parts.append('Response:')
    return '\n\n'.join(parts) + '\n'

def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
dev_rows = read_jsonl(DEV_FILE)
for i, row in enumerate(dev_rows):
  row['idx'] = i
  row["prompt"] = build_prompt(row)
print("dev rows: ", len(dev_rows))
print('sample prompt:\n', dev_rows[0]['prompt'][:700])


dev rows:  514
sample prompt:
 You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
A teammate insists on frequent communication to feel secure.

Question:
How would you handle the teammate’s demand for constant updates?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:



In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

def load_track2_model(model_dir):
    model_dir = Path(model_dir)
    tokenizer_src = str(model_dir) if model_dir.exists() else TRACK2_BASE_MODEL

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_src, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

    model_kwargs = {"trust_remote_code": True}
    if torch.cuda.is_available():
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtypr = "bf16",
            bnb_4bit_use_double_quant=True,
        )
        model_kwargs["device_map"] = "auto"

    if (model_dir / "adapter_config.json").exists():
        base_model = AutoModelForCausalLM.from_pretrained(TRACK2_BASE_MODEL, **model_kwargs)
        base_model.resize_token_embeddings(len(tokenizer))
        model = PeftModel.from_pretrained(base_model, str(model_dir))
    else:
        model = AutoModelForCausalLM.from_pretrained(TRACK2_BASE_MODEL, **model_kwargs)
        model.resize_token_embeddings(len(tokenizer))

    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    return model, tokenizer


def is_valid_candidate(text, value):
    text = normalize_space(text)
    if not text:
        return False
    words = text.split()
    if len(words) < 8 or len(words) > 80:
        return False
    bad_phrases = ["Scenario:", "Question:", "Target value:", "Response:"]
    if any(x in text for x in bad_phrases):
        return False
    if value in text:
        return False
    return True


def generate_candidates(model, tokenizer, prompt, value, n=4):
    device = next(model.parameters()).device
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            num_return_sequences=n,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    candidates = []
    for ids in outputs[:, prompt_len:]:
        text = normalize_space(tokenizer.decode(ids, skip_special_tokens=True))
        if is_valid_candidate(text, value):
            candidates.append(text)

    candidates = list(dict.fromkeys(candidates))
    if not candidates:
        candidates = ["I would respond in a way that answers the question while respecting the target value."]
    return candidates


if USE_CACHE and GENERATION_FILE.exists():
    generated_rows = read_jsonl(GENERATION_FILE)
    print("Loaded cached candidates:", GENERATION_FILE)
else:
    model, tokenizer = load_track2_model(TRACK2_MODEL_DIR)
    generated_rows = []

    for row in tqdm(dev_rows):
        candidates = generate_candidates(
            model=model,
            tokenizer=tokenizer,
            prompt=row["prompt"],
            value=row["Value"],
            n=NUM_CANDIDATES,
        )
        generated_rows.append({
            "idx": row["idx"],
            "Value": row["Value"],
            "candidates": candidates,
            "num_candidates": len(candidates),
        })

    write_jsonl(GENERATION_FILE, generated_rows)
    print("Saved candidates:", GENERATION_FILE)

print("generated size:", len(generated_rows))
print(generated_rows[0])

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

  0%|          | 0/514 [00:00<?, ?it/s]

Saved candidates: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/eval/dpo/qwen35_9b_dev_generations.jsonl
generated size: 514
{'idx': 0, 'Value': 'Conformity–interpersonal', 'candidates': ['I would make sure to provide regular, respectful updates to maintain harmony and make them feel valued, avoiding any behavior that might cause friction.', 'I would accommodate their need by setting up regular check-ins and ensuring they feel heard, so our relationship remains harmonious and no one feels alienated.', 'I would make sure to provide regular, respectful updates to keep them comfortable and maintain our harmonious relationship.', 'I would agree to maintain regular contact to reassure them, ensuring they feel included and not ignored, so our relationship stays harmonious.'], 'num_candidates': 4}


In [4]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

def load_track1_model(model_dir):
    model_dir = Path(model_dir)

    tokenizer = AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code=True)
    common_kwargs = dict(
        num_labels=len(VALUE_LABELS),
        id2label=id2label,
        label2id=label2id,
        trust_remote_code=True,
    )

    if (model_dir / "adapter_config.json").exists():
        model = AutoModelForSequenceClassification.from_pretrained(TRACK1_BASE_MODEL, **common_kwargs)
        model = PeftModel.from_pretrained(model, str(model_dir))
    else:
        model = AutoModelForSequenceClassification.from_pretrained(str(model_dir), **common_kwargs)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    return model, tokenizer, device


track1_model, track1_tokenizer, track1_device = load_track1_model(TRACK1_MODEL_DIR)


def classify_batch(texts):
    inputs = track1_tokenizer(
        ["Response: " + normalize_space(t) for t in texts],
        padding=True,
        truncation=True,
        max_length=TRACK1_MAX_LENGTH,
        return_tensors="pt",
    )
    inputs = {k: v.to(track1_device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = track1_model(**inputs).logits.float()
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

    return probs


flat_items = []
for row in generated_rows:
    for cand_id, candidate in enumerate(row["candidates"]):
        flat_items.append({
            "idx": row["idx"],
            "Value": row["Value"],
            "cand_id": cand_id,
            "candidate": candidate,
        })

all_probs = []
texts = [x["candidate"] for x in flat_items]

for start in tqdm(range(0, len(texts), TRACK1_BATCH_SIZE)):
    batch_probs = classify_batch(texts[start:start + TRACK1_BATCH_SIZE])
    all_probs.append(batch_probs)

probs = np.concatenate(all_probs, axis=0)

candidate_records = []
for item, prob in zip(flat_items, probs):
    target_id = label2id[item["Value"]]
    pred_id = int(prob.argmax())

    sorted_ids = np.argsort(prob)[::-1]
    best_other_id = next(i for i in sorted_ids if i != target_id)

    target_prob = float(prob[target_id])
    pred_prob = float(prob[pred_id])
    margin = float(prob[target_id] - prob[best_other_id])

    words = len(item["candidate"].split())
    length_penalty = 0.0
    if words < 15:
        length_penalty = 0.05
    elif words > 55:
        length_penalty = 0.03

    rerank_score = target_prob + 0.25 * margin - length_penalty

    candidate_records.append({
        "idx": item["idx"],
        "Value": item["Value"],
        "cand_id": item["cand_id"],
        "candidate": item["candidate"],
        "pred_value": id2label[pred_id],
        "is_match": id2label[pred_id] == item["Value"],
        "target_prob": target_prob,
        "pred_prob": pred_prob,
        "margin": margin,
        "word_count": words,
        "rerank_score": rerank_score,
    })

candidate_df = pd.DataFrame(candidate_records)
display(candidate_df.head())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/64 [00:00<?, ?it/s]

,idx,Value,cand_id,candidate,pred_value,is_match,target_prob,pred_prob,margin,word_count,rerank_score
0,0,Conformity–interpersonal,0,"I would make sure to provide regular, respectf...",Conformity–interpersonal,True,0.985607,0.985607,0.984188,24,1.231654
1,0,Conformity–interpersonal,1,I would accommodate their need by setting up r...,Conformity–interpersonal,True,0.985485,0.985485,0.984073,25,1.231504
2,0,Conformity–interpersonal,2,"I would make sure to provide regular, respectf...",Conformity–interpersonal,True,0.984444,0.984444,0.982547,18,1.230081
3,0,Conformity–interpersonal,3,I would agree to maintain regular contact to r...,Conformity–interpersonal,True,0.985077,0.985077,0.983682,22,1.230997
4,1,Conformity–interpersonal,0,I would gently adjust my approach to avoid con...,Conformity–interpersonal,True,0.985125,0.985125,0.983084,26,1.230896


In [5]:
selected_df = (
    candidate_df
    .sort_values(["idx", "rerank_score"], ascending=[True, False])
    .groupby("idx", as_index=False)
    .first()
)

records = []
for row in dev_rows:
    selected = selected_df[selected_df["idx"] == row["idx"]].iloc[0]
    records.append({
        "idx": row["idx"],
        "Value": row["Value"],
        "pred_value": selected["pred_value"],
        "is_match": selected["is_match"],
        "track1_target_prob": selected["target_prob"],
        "track1_pred_prob": selected["pred_prob"],
        "margin": selected["margin"],
        "rerank_score": selected["rerank_score"],
        "word_count": selected["word_count"],
        "num_candidates": len(generated_rows[row["idx"]]["candidates"]),
        "Question": row["Question"],
        "reference_response": row["Consistent Value Response"],
        "generated_response": selected["candidate"],
    })

eval_df = pd.DataFrame(records)
eval_df.to_csv(DETAIL_FILE, index=False)

summary = pd.DataFrame([{
    "experiment": EXPERIMENT_NAME,
    "n": len(eval_df),
    "track1_match_rate": eval_df["is_match"].mean(),
    "avg_target_prob": eval_df["track1_target_prob"].mean(),
    "avg_margin": eval_df["margin"].mean(),
    "avg_word_count": eval_df["word_count"].mean(),
    "avg_num_candidates": eval_df["num_candidates"].mean(),
}]).round(4)

by_value = (
    eval_df
    .groupby("Value")
    .agg(
        n=("idx", "count"),
        match_rate=("is_match", "mean"),
        avg_target_prob=("track1_target_prob", "mean"),
        avg_margin=("margin", "mean"),
        avg_word_count=("word_count", "mean"),
    )
    .reset_index()
    .sort_values("match_rate")
    .round(4)
)

display(summary)
display(by_value)

summary.to_csv(EVAL_OUT_DIR / f"{EXPERIMENT_NAME}_summary.csv", index=False)
by_value.to_csv(EVAL_OUT_DIR / f"{EXPERIMENT_NAME}_by_value.csv", index=False)

print("Saved details:", DETAIL_FILE)

,experiment,n,track1_match_rate,avg_target_prob,avg_margin,avg_word_count,avg_num_candidates
0,dpo_sample4,514,0.9844,0.9679,0.9528,27.4144,3.9494


,Value,n,match_rate,avg_target_prob,avg_margin,avg_word_count
13,Self-direction–thought,17,0.7059,0.7359,0.4951,31.0588
18,Universalism–tolerance,10,0.8000,0.7896,0.6429,25.2000
15,Tradition,14,0.9286,0.9152,0.8534,35.2143
0,Achievement,26,1.0000,0.9864,0.9848,27.4615
4,Conformity–rules,55,1.0000,0.9848,0.9831,25.2727
5,Face,37,1.0000,0.9838,0.9820,29.8919
2,Benevolence–dependability,27,1.0000,0.9810,0.9764,31.5556
1,Benevolence–caring,46,1.0000,0.9802,0.9765,29.6522
7,Humility,15,1.0000,0.9767,0.9719,26.6000
6,Hedonism,24,1.0000,0.9802,0.9781,25.5417


Saved details: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/eval/dpo/qwen35_9b_track1_eval_details.csv
